# 09 — 輪郭検出・形状特徴

## 目標
- `cv2.findContours` で輪郭を検出
- バウンディングボックス / 最小外接円 / 最小外接矩形を求める
- `cv2.moments` で重心・面積を計算
- `cv2.approxPolyDP` で頂点を近似

## モード
- `RETR_TREE`: 全階層を取得 (親子関係あり)
- `CHAIN_APPROX_SIMPLE`: 冗長な中間点を省略

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import cv2
import numpy as np
from utils import DATA_DIR, load_image, show_nb

src  = load_image(DATA_DIR / 'lena.png')
gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
_, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
contours, _ = cv2.findContours(binary, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
contours = [c for c in contours if cv2.contourArea(c) > 100]
print(f'輪郭数: {len(contours)}')

In [ ]:
all_cont = src.copy()
cv2.drawContours(all_cont, contours, -1, (0, 255, 0), 2)

bbox_img = src.copy()
for c in contours:
    x, y, w, h = cv2.boundingRect(c)
    cv2.rectangle(bbox_img, (x, y), (x+w, y+h), (255, 0, 0), 2)

circle_img = src.copy()
for c in contours:
    (cx, cy), r = cv2.minEnclosingCircle(c)
    cv2.circle(circle_img, (int(cx), int(cy)), int(r), (0, 0, 255), 2)

show_nb([
    ('binary', binary),
    ('all contours', all_cont),
    ('bounding box', bbox_img),
    ('min circle', circle_img),
], cols=2)

In [ ]:
# 最大輪郭の面積・重心・近似
largest = max(contours, key=cv2.contourArea)
M = cv2.moments(largest)
cx = int(M['m10'] / M['m00'])
cy = int(M['m01'] / M['m00'])
print(f'面積: {cv2.contourArea(largest):.1f}')
print(f'重心: ({cx}, {cy})')
print(f'周長: {cv2.arcLength(largest, True):.1f}')

approx_img = src.copy()
for c in contours:
    eps = 0.02 * cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, eps, True)
    cv2.drawContours(approx_img, [approx], 0, (0, 128, 255), 2)
show_nb([('approx poly', approx_img)], cols=1)